# Experiment 2 — `repartition(n)` vs `coalesce(n)`

Same pipeline, one call changed. Spark UI: http://localhost:4040 → **Jobs**, **Stages**.

In [1]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m3-exp2-repartition-coalesce").getOrCreate()
big = spark.read.parquet("/opt/data/m3/orders_big")
print("input partitions:", big.rdd.getNumPartitions())

def timed(label, df, path):
    t0 = time.perf_counter()
    df.write.mode("overwrite").parquet(path)
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

filtered = big.filter(F.col("amount") > 400)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/26 12:38:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


input partitions: 16


## A — `coalesce(4)` before the write

**PREDICT:** jobs; stages; tasks in each stage; whether a shuffle happens; number of output files.

In [2]:
filtered.coalesce(4).explain("formatted")

== Physical Plan ==
Coalesce (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(amount), GreaterThan(amount,400.0)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]

(3) Filter [codegen id : 1]
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : (isnotnull(amount#3) AND (amount#3 > 400.0))

(4) Coalesce
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: 4




In [3]:
timed("A coalesce(4)", filtered.coalesce(4), "/opt/data/m3/out/coalesce4")

[Stage 1:=============================>                             (2 + 2) / 4]

A coalesce(4): 3.4s


## B — `repartition(4)` before the write

**PREDICT:** the same four things, and the runtime relative to A.

In [4]:
filtered.repartition(4).explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (4)
+- Exchange (3)
   +- Filter (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(amount), GreaterThan(amount,400.0)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : (isnotnull(amount#3) AND (amount#3 > 400.0))

(3) Exchange
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=74]

(4) AdaptiveSparkPlan
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: isFinalPlan=false




In [5]:
timed("B repartition(4)", filtered.repartition(4), "/opt/data/m3/out/repartition4")

[Stage 4:============================================>              (3 + 1) / 4]

B repartition(4): 2.3s


## C — asking for *more* partitions than the input has

**PREDICT:** the partition count each call reports.

In [6]:
print("coalesce(32):   ", big.coalesce(32).rdd.getNumPartitions())
print("repartition(32):", big.repartition(32).rdd.getNumPartitions())

coalesce(32):    16


[Stage 5:=============================>                            (8 + 8) / 16]

repartition(32): 32


## Observe

Jobs and Stages tabs for A and B: which stage did the filtering and how many tasks it had; shuffle write / read; durations. Count the files under each output path:

In [7]:
for p in ["coalesce4", "repartition4"]:
    print(p, len(spark.read.parquet(f"/opt/data/m3/out/{p}").inputFiles()), "files")

coalesce4 4 files
repartition4 4 files


## Stop before opening the next notebook

In [ ]:
spark.stop()